### Q1. First trace

In [1]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import (
    ConsoleSpanExporter,
    SimpleSpanProcessor,
)

provider = TracerProvider()

provider.add_span_processor(
    SimpleSpanProcessor(ConsoleSpanExporter())
)

trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

In [2]:
from starter import rag
from starter import RAGBase
from opentelemetry import trace

tracer = trace.get_tracer("llm-zoomcamp")


class RAGTraced(RAGBase):

    def rag(self, query):
        print("rag() called")

        with tracer.start_as_current_span("rag"):

            search_results = self.search(query)

            print("after search")

            prompt = self.build_prompt(
                query,
                search_results
            )

            print("before llm")

            response = self.llm(prompt)

            print("after llm")

            return response.output_text


    def search(self, query):
        print("search() called")

        with tracer.start_as_current_span("search"):
            return super().search(query)


    def llm(self, prompt):
        print("llm() called")

        with tracer.start_as_current_span("llm"):
            return super().llm(prompt)

In [3]:
traced_rag = RAGTraced(
    index=rag.index,
    llm_client=rag.llm_client
)

In [4]:
query = "How does the agentic loop keep calling the model until it stops?"

answer = traced_rag.rag(query)

print(answer)

rag() called
search() called
{
    "name": "search",
    "context": {
        "trace_id": "0x0a68ca02fef694283d8e7776bff99f6b",
        "span_id": "0xb0f832694f9d2848",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xe0cd74fbac3ab3c2",
    "start_time": "2026-07-21T17:30:06.957146Z",
    "end_time": "2026-07-21T17:30:06.960145Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "8368b220-9c5c-4de9-83aa-cb10a8a2b19a",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
after search
before llm
llm() called
{
    "name": "llm",
    "context": {
        "trace_id": "0x0a68ca02fef694283d8e7776bff99f6b",
        "span_id": "0x7e46b7c60